# Assignment 3: Image Generation

## GAN Architecture

In [ ]:
import torch
from helper_lib.model import get_model

generator, discriminator = get_model('GAN')
noise = torch.randn(8, 100)
fake_images = generator(noise)
print('Generator output:', fake_images.shape)
print('Discriminator output:', discriminator(fake_images).shape)


## MNIST Training

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from helper_lib.trainer import train_gan
import torch.nn as nn
import torch.optim as optim

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator, discriminator = get_model('GAN')
generator = generator.to(device)
discriminator = discriminator.to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))


In [ ]:
generator, discriminator = train_gan(
    (generator, discriminator), train_loader, criterion,
    (optimizer_G, optimizer_D), device=device, epochs=10
)

torch.save(generator.cpu().state_dict(), 'gan_generator.pth')


In [ ]:
from helper_lib.generator import generate_samples

fig = generate_samples(generator, device='cpu', num_samples=16)
fig.savefig('generated_digits.png')


## API

The GAN uses `get_model` in `helper_lib/model.py`, `train_gan` in `helper_lib/trainer.py`, and `generate_samples` in `helper_lib/generator.py`.
After training, keep `gan_generator.pth` beside `gan_api.py`. The endpoint is added to the existing `app/main.py`.
Run `python -m uvicorn app.main:app --reload` and open `http://127.0.0.1:8000/generate-gan`.


## Theory: Building Blocks of GAN

**Question 1**

The transposed convolution formula is

\[
O=(I-1)S-2P+K+OP
\]

With $I=8$, $K=4$, $S=2$, $P=1$, and $OP=1$:

\[
O=(8-1)(2)-2(1)+4+1=17
\]

The output size is **17 × 17**.

**Question 2**

If the stride changes to 3:

\[
O=(8-1)(3)-2(1)+4+1=24
\]

The output changes from 17 × 17 to **24 × 24**. Each side increases by 7.

**Question 3**

For each spatial dimension, the formula is

\[
\boxed{O=(I-1)S-2P+K+OP}
\]

**Question 4**

One possible choice is **kernel size 2 × 2, stride 2, padding 0, output padding 0**.

\[
O=(16-1)(2)-0+2+0=32
\]

So the result is 32 × 32.

**Question 5**

Values: $[6,8,10,6]$.

Mean:

\[
\mu=\frac{6+8+10+6}{4}=7.5
\]

Variance (using the mini-batch size):

\[
\sigma^2=\frac{(6-7.5)^2+(8-7.5)^2+(10-7.5)^2+(6-7.5)^2}{4}
=\frac{11}{4}=2.75
\]

Standard deviation: $\sigma=\sqrt{2.75}\approx1.6583$.

Normalized values (ignoring the small numerical $\epsilon$):

\[
\frac{x-\mu}{\sigma}\approx[-0.9045,\ 0.3015,\ 1.5076,\ -0.9045]
\]

**Question 6**

ReLU:

\[
f(x)=\max(0,x)=\begin{cases}x&x>0\\0&x\leq0\end{cases}
\]

LeakyReLU:

\[
f(x)=\begin{cases}x&x>0\\\alpha x&x\leq0\end{cases}
\]

Here $\alpha$ is a small positive number, such as $0.2$. Unlike ReLU, LeakyReLU does not make every negative input zero.

**Question 7**

LeakyReLU keeps a small gradient for negative inputs. ReLU has zero gradient for negative inputs, so some neurons may stop learning. LeakyReLU can reduce this problem in deep networks.